<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/LangChain_Chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detailed Guide to LangChain Chains

In LangChain, a **Chain** is an abstract concept representing a sequence of calls to a components (such as prompt templates, LLMs, output parsers, or other chains). It allows you to combine multiple components into a single, cohesive application.

Historically, LangChain relied on explicit subclasses like `LLMChain`, `SimpleSequentialChain`, etc. (legacy chains). Today, the standard and recommended way to build chains is using **LCEL (LangChain Expression Language)**, which uses the pipe operator (`|`) to compose components together seamlessly.

Let's break down the main concepts:
1. **Legacy Chains vs. LCEL**
2. **How LCEL Works (Runnable Protocol)**
3. **Practical Examples**

### 1. Legacy Chains vs. LCEL

* **Legacy Chains (e.g., `LLMChain`, `SequentialChain`):** These were pre-built class structures. While easy to use at first, they were hard to customize, did not natively support streaming or async operations consistently, and obscured what was happening under the hood.
* **LCEL (LangChain Expression Language):** A declarative way to chain LangChain components together. Any component that implements the `Runnable` protocol can be chained using the `|` operator. It offers built-in support for streaming, async operations, parallel execution, and batch processing.

### 2. Basic Setup and LCEL Chain Example

Let's install the required packages and build a classic chain: `PromptTemplate -> ChatModel -> OutputParser` using LCEL.

In [15]:
!pip install -q langchain langchain-core langchain-openai

### 3. Advanced Chain Types

Using LCEL, you can build complex chains:

* **Sequential Chains:** The output of one chain becomes the input to another. This is achieved by feeding the result of one runnable into another using a dictionary mapping or helper runnables like `RunnablePassthrough`.
* **Router/Conditional Chains:** Dynamically route the input to different sub-chains based on conditions (e.g., classifying a user query as 'billing' or 'tech support' and sending it to the respective chain).
* **RetrievalQA Chains (RAG):** Combine a Vector Store retriever, a formatting step, a prompt, and a model to answer questions based on external documents.

In [16]:

%pip install -q -U langchain langchain-core langchain-openai


In [17]:

from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough
)

api_key = userdata.get("NVIDIA_API_KEY")

llm = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
    temperature=0.4,
    max_retries=3,
    timeout=60
)

parser = StrOutputParser()

print("NVIDIA LLM configured successfully!")


NVIDIA LLM configured successfully!


## Comprehensive Guide to LCEL Chain Types

Now, let's look at how to construct and understand different types of chains step-by-step, using the `llm` and `parser` instances we just created.

### 1. Simple Chain
A **Simple Chain** is a straightforward sequence where a prompt template feeds directly into the language model, and its output is formatted by an output parser.

`PromptTemplate` ➔ `LLM` ➔ `OutputParser`

In [18]:
# 1. Simple Chain
prompt_simple = ChatPromptTemplate.from_template("What is a unique fact about the animal: {animal}?")

simple_chain = prompt_simple | llm | parser

# Let's run it
fact_result = simple_chain.invoke({"animal": "platypus"})
print(fact_result)

Here's a uniquely distinctive fact about the platypus:

**Male platypuses have venomous spurs on their hind legs.** 

The venom is produced in crural glands and delivered through a sharp spur on each ankle. It's strong enough to cause severe pain, swelling, and prolonged hypersensitivity in humans, and is used primarily during breeding season to compete with other males. Among mammals, true venom delivery is extremely rare—only a handful of species (like the male platypus and the male echidna) evolved this trait, making it one of the most unusual mammalian defenses in the animal kingdom.

(Other equally unique traits: they lay eggs despite being mammals, have no stomach—their esophagus connects directly to their intestine—and males possess 10 sex chromosomes instead of the typical X/Y pair.)


# Program

In [19]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt=ChatPromptTemplate.from_template(
    "Explain {topic} with in simple language with one program and example"
)

chain=prompt | llm | StrOutputParser()

result=chain.invoke({"topic": "Python"})
print(result)

**Python** is a programming language built to be easy to read and write. It uses plain English-like commands, doesn't require fussy symbols (like semicolons at the end of lines), and uses indentation (spaces/tabs) to organize code. It's used for websites, data science, automation, games, and more.

### 👨‍💻 One Simple Program

```python
name = "Python Learner"
age = 10
print(f"Hello! I'm {name} and I'm learning Python at age {age}.")
```

### 🧾 How It Works (Line by Line)

1. **`name = "Python Learner"`**  
   Creates a labeled container called `name` and puts the text `"Python Learner"` inside it.

2. **`age = 10`**  
   Creates another container called `age` and puts the number `10` inside it.

3. **`print(...)`**  
   This is a built-in command that shows text on the screen. The `f"..."` (called an f-string) lets you slip the variables inside the sentence automatically.

### 📦 What You Get When You Run It

```
Hello! I'm Python Learner and I'm learning Python at age 10.
```

### 💡 Qu

### 2. Two-Step Chain & Three-Step Chain (Sequential Design)
Sequential chains pass the output of one step as the input to the next step.

* **Two-Step Chain:**
  * **Step 1:** Generate a concept definition.
  * **Step 2:** Translate that definition into French.
* **Three-Step Chain:**
  * **Step 1:** Generate a concept definition.
  * **Step 2:** Translate that definition into French.
  * **Step 3:** Summarize the French translation into a single sentence.

In [20]:
# Step Templates
prompt_step1 = ChatPromptTemplate.from_template("Explain the concept of {concept} in 1 short paragraph.")
prompt_step2 = ChatPromptTemplate.from_template("Translate the following text to French:\n\n{text}")
prompt_step3 = ChatPromptTemplate.from_template("Summarize this French text in one sentence:\n\n{french_text}")

# Building the Two-Step Chain
# We use RunnablePassthrough/RunnableParallel or dictionary mapping to feed outputs forward
two_step_chain = (
    prompt_step1
    | llm
    | parser
    | {"text": RunnablePassthrough()}
    | prompt_step2
    | llm
    | parser
)

# Building the Three-Step Chain
three_step_chain = (
    two_step_chain
    | {"french_text": RunnablePassthrough()}
    | prompt_step3
    | llm
    | parser
)

print("--- Running Two-Step Chain (Translation) ---")
translated_text = two_step_chain.invoke({"concept": "Quantum Computing"})
print(translated_text)

print("\n--- Running Three-Step Chain (Summary of Translation) ---")
final_summary = three_step_chain.invoke({"concept": "Quantum Computing"})
print(final_summary)

--- Running Two-Step Chain (Translation) ---
L'informatique quantique est un paradigme de calcul qui s'appuie sur les principes de la mécanique quantique—principalement la superposition et l'entanglement—pour traiter l'information à l'aide de bits quantiques (qubits) plutôt que de bits classiques. Alors que les bits classiques sont strictement 0 ou 1, les qubits peuvent exister dans une superposition des deux états simultanément, et les qubits en état d'entanglement peuvent correlérer de manière à permettre aux ordinateurs quantiques d'explorer de vastes espaces de solutions en parallèle. Cela leur permet de résoudre des problèmes complexes spécifiques—tels que la cryptographie, la simulation moléculaire et l'optimisation—exponentiellement plus vite que même les supercalculateurs classiques les plus puissants, bien que la technologie demeure largement expérimentale, hautement sujette aux erreurs et pas encore adaptée aux tâches générales.

--- Running Three-Step Chain (Summary of Trans

# Program

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Step 1: Generate the explanation in English
prompt1 = ChatPromptTemplate.from_template("""
Explain the topic "{topic}" in detail in English.

Include:
- Introduction
- Basic facts
- Important contributions
- Key concepts
- Examples where relevant

Provide a complete, well-structured explanation.
""")

# Step 2: Translate the COMPLETE explanation into Urdu
prompt2 = ChatPromptTemplate.from_template("""
You are an expert English-to-Urdu translator.

Translate the ENTIRE text below into natural, fluent Urdu.

STRICT RULES:
1. Translate every sentence and paragraph.
2. Do not summarize, shorten, or omit any information.
3. Translate all headings, subheadings, bullet points, and examples.
4. Use Urdu script (اردو), NOT Roman Urdu.
5. Translate English words into Urdu wherever appropriate.
6. Keep proper names, dates, and technical terms accurate.
7. Preserve the original structure and meaning.
8. Return only the complete Urdu translation.

TEXT TO TRANSLATE:
{text}
""")

parser = StrOutputParser()

# Step 3: Build the two-step chain
two_step_chain = (
    prompt1
    | llm
    | parser
    | RunnableLambda(lambda text: {"text": text})
    | prompt2
    | llm
    | parser
)

# Step 4: Run the chain
result = two_step_chain.invoke({
    "topic": "Imam Ahmad Raza Khan"
})

print(result)

# امام احمد رضا خان: ایک جامع総合ビュー

## مقدمہ
امام احمد رضا خان (1856–1921)، جو عوامی طور پر **الا Hazrat** یا **امامؔ اهل السّنۃ** کے نام سے مشہور ہیں، late colonial India کے Sab se prolific aur influential Islamic scholars میں سے ایک تھے۔ وہ Hanafi jurist، classical Islamic scholar، poet، اور muhaddith (hadith master) Bareilly Sharif میں مبنی تھے (present-day Uttar Pradesh)۔ وہ Barelvi movement کے numainda figure تھے، لیکن خود **Ahl al-Sunnah wa al-Jama'ah** (Sunnat aur Jamaat کے log) کے titre کو پیش کرنا پسند کرتے تھے۔ Apne zindagi ka kaam primarily traditional Sunni Islam ke reformist currents against defence تھا، Islamic law ka systematization tha، aur Prophet Muhammad ke liye devotional love ka revival tha۔ Apne mausoleum ka Bareilly main miliionوں followers across South Asia aur diaspora ke liye major pilgrimage site bana hai.

## بنیادی faits
- **پورا نام:** Ahmad Raza Khan Barelvi (bhi written as Ahmed Raza Khan)
- **پیدائش:** 14 June 1856، Patna, Bengal Presidency (present-day

In [22]:
# Three-Step Chain: Quran Surah → Tafseer → Hadith

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Use your existing NVIDIA LLM initialization
# Example:
# from langchain_openai import ChatOpenAI
# import os
# from dotenv import load_dotenv
#
# load_dotenv()
#
# llm = ChatOpenAI(
#     model="YOUR_NVIDIA_MODEL_NAME",
#     base_url="https://integrate.api.nvidia.com/v1",
#     api_key=os.getenv("NVIDIA_API_KEY"),
#     temperature=0.3
# )

parser = StrOutputParser()

# STEP 1: Explain the Surah
prompt1 = ChatPromptTemplate.from_template("""
You are a knowledgeable Quran educator.

Explain Surah {surah} in simple, clear language from Kanzul Iman which was written by Imam Ahmad Raza Khan.

Include:
1. Introduction and meaning of the Surah's name
2. Whether it is Makki or Madani, if reliably established
3. Number of verses
4. Main themes and lessons
5. Historical context, where supported by reliable sources

Avoid presenting disputed information as certain.
Clearly distinguish established facts from scholarly differences.

Write the explanation in English.
""")

# STEP 2: Provide Tafseer
prompt2 = ChatPromptTemplate.from_template("""
You are an assistant helping readers understand Quranic Tafseer.

Surah: {surah}

Original explanation:
{explanation}

Now provide a structured Tafseer of this Surah.

Include:
1. Explanation of the main verses or passages
2. Relevant linguistic meanings
3. Context of revelation (Asbab al-Nuzul), only where reliably reported
4. Interpretations from recognized Tafseer scholarship
5. Practical lessons

Do not invent Quranic verses, quotations, or scholarly references.
Mention differences of interpretation when relevant.
Write in English and clearly identify uncertain reports.
""")

# STEP 3: Provide relevant Hadith
prompt3 = ChatPromptTemplate.from_template("""
You are an assistant helping readers study Hadith in relation to the Quran.

Surah: {surah}

Original explanation:
{explanation}

Tafseer:
{tafseer}

Now provide Hadith relevant to the themes, teachings, or virtues of this Surah.

For each narration, include:
1. Hadith text or a faithful translation
2. The connection to the Surah
3. Collection name, such as Sahih al-Bukhari or Sahih Muslim,
   where applicable
4. Hadith number or book/chapter reference when verifiable
5. Authenticity grading, only when supported by a reliable reference

Important:
- Do not fabricate Hadith or references.
- Do not claim that a Hadith is specifically about this Surah
  unless reliable evidence establishes that connection.
- If you cannot verify a relevant narration, say so clearly.
- Distinguish general thematic relevance from narrations
  specifically mentioning the Surah.

Write in English.
""")

# BUILD THE THREE-STEP CHAIN

three_step_chain = (
    prompt1
    | llm
    | parser
    | RunnableLambda(lambda explanation: {
        "surah": surah_name,
        "explanation": explanation
    })
    | prompt2
    | llm
    | parser
    | RunnableLambda(lambda tafseer: {
        "surah": surah_name,
        "explanation": original_explanation,
        "tafseer": tafseer
    })
    | prompt3
    | llm
    | parser
)

# RUN THE PROGRAM

surah_name = input("Enter the Surah name: ")

# Generate the first step separately so its output is retained
explanation = (
    prompt1
    | llm
    | parser
).invoke({"surah": surah_name})

print("\n" + "=" * 60)
print("STEP 1: SURAH EXPLANATION")
print("=" * 60)
print(explanation)

# Generate Tafseer using the explanation from Step 1
tafseer = (
    prompt2
    | llm
    | parser
).invoke({
    "surah": surah_name,
    "explanation": explanation
})

print("\n" + "=" * 60)
print("STEP 2: TAFSEER")
print("=" * 60)
print(tafseer)

# Generate Hadith using both previous outputs
hadith = (
    prompt3
    | llm
    | parser
).invoke({
    "surah": surah_name,
    "explanation": explanation,
    "tafseer": tafseer
})

print("\n" + "=" * 60)
print("STEP 3: RELATED HADITH")
print("=" * 60)
print(hadith)

Enter the Surah name: Yusuf

STEP 1: SURAH EXPLANATION
**Surah Yusuf (سورة Yusuf)**  
*The Eleventh Chapter of the Quran – A Beacon of Patience, Trust, and Divine Wisdom*

### 1. Introduction and Meaning of the Surah's Name
Surah Yusuf is the 12th chapter of the Quran and is named after **Prophet Yusuf (Joseph, AS)**, whose inspiring life story forms the core of the surah. In Arabic, *Yusuf* is the name itself; the title identifies the prophet around whom the chapter revolves. It is one of the most detailed and beautifully narrated stories in the Quran, often called the "essence" or "peak" of the Quranic stories due to its depth, emotional resonance, and comprehensive lessons.

### 2. Whether it is Makki or Madani – Reliably Established
**Established Fact:** Surah Yusuf is overwhelmingly classified as a **Makki** surah, meaning it was revealed before the Hijrah to Medina, during the Prophet’s mission in Mecca.  
**Scharly Note:** The vast majority of classical and contemporary scholars

### 3. Sequential & Parallel Chains (Multi-Input/Output)
Sometimes you need to process inputs in parallel (e.g., generate a pros and cons list simultaneously) before combining them into a final synthesis.

In [23]:
pros_prompt = ChatPromptTemplate.from_template("List 2 major advantages of {technology}.")
cons_prompt = ChatPromptTemplate.from_template("List 2 major disadvantages of {technology}.")

synthesis_prompt = ChatPromptTemplate.from_template(
    "Given these pros:\n{pros}\n\nAnd these cons:\n{cons}\n\nWrite a balanced summary recommendation."
)

# Executing step 1 (pros and cons) in parallel
parallel_branch = RunnableParallel(
    pros=(pros_prompt | llm | parser),
    cons=(cons_prompt | llm | parser)
)

# Complete multi-chain pipeline
multi_chain_app = parallel_branch | synthesis_prompt | llm | parser

recommendation = multi_chain_app.invoke({"technology": "autonomous driving cars"})
print(recommendation)

Autonomous vehicles hold significant promise for transforming road safety and traffic flow, potentially reducing crashes caused by human error and easing congestion through vehicle-to-infrastructure communication. However, substantial barriers must be addressed before these benefits can be reliably realized.

On the technology front, current systems struggle with "long-tail" edge cases–unpredictable human driver behavior, extreme weather, unmapped roadwork, debris, and complex urban intersections–where sensor suites and AI decision-making frequently fail, raising safety concerns in conditions where human drivers remain more dependable. Parallel to this, ethical, legal, and liability uncertainties loom large: it is unclear how vehicles should be programmed for moral dilemmas, who is legally responsible in a crash, or how existing insurance, criminal, and civil frameworks will adapt across jurisdictions, creating regulatory hurdles and public trust barriers that slow deployment.

A balan

# Program 1 — Simple sequential chain

In [25]:
prompt1 = ChatPromptTemplate.from_template(
    "Explain {topic} in simple English with examples."
)

prompt2 = ChatPromptTemplate.from_template(
    "Summarize the following explanation in 5 bullet points:\n\n{text}"
)

explanation_chain = prompt1 | llm | parser

summary_chain = (
    RunnablePassthrough.assign(
        text=explanation_chain
    )
    | prompt2
    | llm
    | parser
)

result = summary_chain.invoke({"topic": "Machine Learning"})

print(result)

- Machine Learning lets computers improve at tasks by finding patterns in data, much like a child learns to recognize a dog from many examples.  
- The core process involves feeding data to a computer, which autonomously searches for patterns, makes predictions, and refines its performance through feedback and adjustment.  
- The three main types are supervised learning (learns from labeled examples), unsupervised learning (finds hidden structures in unlabeled data), and reinforcement learning (learns via trial-and-error rewards and penalties).  
- Everyday applications include spam filters, Netflix/Spotify recommendations, Google Translate, self-driving cars, and voice assistants, all powered by ML learning from vast datasets.  
- ML is not magic but math-driven pattern recognition; accuracy grows with more data, and users provide the information rather than programming explicit rules.


# Program 2 — Three-step sequential chain

In [26]:
prompt1 = ChatPromptTemplate.from_template(
    "Explain {topic} in detail."
)

prompt2 = ChatPromptTemplate.from_template(
    "Summarize this explanation:\n\n{explanation}"
)

prompt3 = ChatPromptTemplate.from_template(
    "Create 5 MCQs based on this explanation and summary.\n"
    "Explanation:\n{explanation}\n\n"
    "Summary:\n{summary}"
)

explanation_chain = prompt1 | llm | parser

summary_chain = (
    prompt2 | llm | parser
)

quiz_chain = prompt3 | llm | parser

three_step_chain = (
    RunnablePassthrough.assign(
        explanation=explanation_chain
    )
    | RunnablePassthrough.assign(
        summary=summary_chain
    )
    | quiz_chain
)

result = three_step_chain.invoke({
    "topic": "Computer Networks"
})

print(result)

### 5 MCQs on Computer Networks (Based on the Provided Explanation & Summary)

**1. In the TCP/IP model, which single layer combines the functions of the OSI model's Session, Presentation, and Application layers?**
A) Transport Layer  
B) Internet Layer  
C) Network Access Layer  
D) Application Layer  

**2. Which protocol is primarily responsible for translating human-readable domain names (e.g., `google.com`) into IP addresses?**
A) HTTP  
B) DNS  
C) DHCP  
D) FTP  

**3. Which cabling type is immune to electromagnetic interference (EMI) and supports the highest speeds over the longest distances?**
A) Twisted Pair (Cat6/6a)  
B) Coaxial Cable  
C) Fiber Optics  
D) Unshielded Twisted Pair (UTP)  

**4. Within the CIA triad of network security, which component ensures that data has not been altered, deleted, or tampered with during transmission?**
A) Confidentiality  
B) Integrity  
C) Availability  
D) Authenticity  

**5. Which modern networking trend involves processing data clos

# Program 3 — Basic parallel chain

In [27]:
prompt1 = ChatPromptTemplate.from_template(
    "Explain {topic} in simple English with examples."
)

prompt2 = ChatPromptTemplate.from_template(
    "List 5 advantages of {topic}."
)

prompt3 = ChatPromptTemplate.from_template(
    "List 5 disadvantages of {topic}."
)

prompt4 = ChatPromptTemplate.from_template(
    "Generate 5 MCQs about {topic} with answers."
)

parallel_chain = RunnableParallel(
    explanation=prompt1 | llm | parser,
    advantages=prompt2 | llm | parser,
    disadvantages=prompt3 | llm | parser,
    quiz=prompt4 | llm | parser
)

result = parallel_chain.invoke({
    "topic": "Artificial Intelligence"
})

print("EXPLANATION:\n", result["explanation"])
print("\nADVANTAGES:\n", result["advantages"])
print("\nDISADVANTAGES:\n", result["disadvantages"])
print("\nQUIZ:\n", result["quiz"])

EXPLANATION:
 **What is AI?**
AI (Artificial Intelligence) is when a computer or machine can do tasks that normally require human smarts: understanding language, recognizing pictures, learning from experience, or making decisions.

**How it works (in a nutshell)**
Show a computer thousands of examples of something, and it starts to notice patterns. After enough practice, it can correctly handle new examples it’s never seen before. It’s a lot like a baby learning what a cat is: after seeing many cats, they notice the pointy ears, whiskers, and fur, and later they can point out a cat in a new picture.

**Everyday examples**
- **Voice assistants** (Siri, Alexa, Google Assistant): You ask "What’s the weather?" and the AI hears your words, figures out what you mean, and gives you an answer.
- **Recommendations** (Netflix, Spotify, YouTube): "Because you watched X, we think you’ll like Y." The AI studies your habits and predicts what you’ll enjoy.
- **Photo apps**: Type "beach" and the app s

# Program 4 — Parallel chain for Quran study

In [28]:
surah_prompt = ChatPromptTemplate.from_template("""
Explain the main themes of Surah {surah}.
Distinguish established facts from uncertain interpretations.
""")

lessons_prompt = ChatPromptTemplate.from_template("""
List important lessons from Surah {surah}.
Explain each lesson briefly.
""")

vocabulary_prompt = ChatPromptTemplate.from_template("""
Explain important Arabic vocabulary from Surah {surah}.
Give accurate meanings and avoid guessing.
""")

hadith_prompt = ChatPromptTemplate.from_template("""
Identify Hadith relevant to the themes of Surah {surah}.
Provide verifiable collection references and authenticity
information when available. Do not invent narrations.
Distinguish general relevance from a specific Surah virtue.
""")

quran_parallel_chain = RunnableParallel(
    themes=surah_prompt | llm | parser,
    lessons=lessons_prompt | llm | parser,
    vocabulary=vocabulary_prompt | llm | parser,
    hadith_research=hadith_prompt | llm | parser
)

result = quran_parallel_chain.invoke({
    "surah": "Al-Fatiha"
})

for section, content in result.items():
    print(f"\n{'=' * 50}")
    print(section.upper())
    print("=" * 50)
    print(content)


THEMES
# Main Themes of Surah Al-Fatiha: Established Facts vs. Uncertain Interpretations

Surah Al-Fatiha (The Opening) is the first chapter of the Quran, consisting of 7 verses. It is recited in every unit of the Islamic prayer (salah) and is variously known as *Umm al-Kitab* (Mother of the Book), *Al-Hamd* (The Praise), and *As-Salah* (in hadith literature, referring to its essential role in prayer). Below are its core themes, divided into what is **textually and theologically established** and what is subject to **scholarly difference or varying interpretation**.

---
## 🔹 Established Facts (Consensus of Text & Classical Tafsir)

These are points grounded in the Arabic text, agreed upon by the majority of classical exegetes (*mufassirun*), and integral to the surah’s liturgical and theological function.

| Theme | Established Detail |
|---|---|
| **Opening doxology** | "Al-hamdu lillahi rabb al-'alamin" — Praise belongs to Allah, Lord of the worlds. This is the fixed opening formul

# Program 5 — Parallel translation into multiple languages

In [29]:
english_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in English."
)

urdu_prompt = ChatPromptTemplate.from_template("""
Translate this topic explanation into complete Urdu script.
Do not summarize or omit information.

Topic: {topic}
""")

hindi_prompt = ChatPromptTemplate.from_template("""
Explain {topic} in Hindi using Devanagari script.
""")

multilingual_chain = RunnableParallel(
    english=english_prompt | llm | parser,
    urdu=urdu_prompt | llm | parser,
    hindi=hindi_prompt | llm | parser
)

result = multilingual_chain.invoke({
    "topic": "Computer Networks"
})

print("ENGLISH:\n", result["english"])
print("\nURDU:\n", result["urdu"])
print("\nHINDI:\n", result["hindi"])

ENGLISH:
 ### What Is a Computer Network?
At its simplest, a **computer network** is a group of two or more devices (called *nodes*) that are connected so they can share data, resources, and communicate. The connections can be made via cables, fiber optics, Wi-Fi, satellite, or other transmission media. Networks range from a few devices in a single room to billions of devices across the globe (the Internet).

---

### Key Components
| Component | Role |
|-----------|------|
| **Nodes/Devices** | Computers, smartphones, servers, printers, IoT devices, switches, routers |
| **Links/Medium** | Ethernet cables, Wi-Fi radio waves, fiber optic lines, coaxial cables, 5G/4G signals |
| **Protocols** | Rulebooks that dictate how data is formatted, addressed, sent, and received (e.g., TCP/IP, HTTP, DNS) |
| **Hardware** | Routers (direct traffic between networks), switches (connect devices locally), access points (extend Wi-Fi), modems (convert signals for internet entry) |

---

### Common Netw

### 4. Conditional (Router) Chain
A **Conditional Chain** dynamically decides which path to execute based on a condition or classification step.

In [24]:
from langchain_core.runnables import RunnableBranch

# Define specialized chains
math_chain = ChatPromptTemplate.from_template("Solve this math problem clearly: {query}") | llm | parser
creative_chain = ChatPromptTemplate.from_template("Write an imaginative story or creative response to: {query}") | llm | parser

# Router step to classify incoming query
router_prompt = ChatPromptTemplate.from_template(
    "Classify the following query into exactly one of these categories: 'math' or 'creative'.\nQuery: {query}\nCategory:"
)
router_chain = router_prompt | llm | parser

# Build the Conditional Chain using RunnableBranch
conditional_chain = RunnableBranch(
    (lambda x: "math" in x["topic"].lower(), math_chain),
    creative_chain # Default branch
)

# Orchestrate classification and routing
full_router_pipeline = (
    RunnableParallel(topic=router_chain, query=lambda x: x["query"])
    | conditional_chain
)

print("--- Routing Math Query ---")
print(full_router_pipeline.invoke({"query": "What is the derivative of x^2 + 5x?"}))

print("\n--- Routing Creative Query ---")
print(full_router_pipeline.invoke({"query": "Write a poem about a robot learning to feel love."}))

--- Routing Math Query ---
To find the derivative of \( f(x) = x^2 + 5x \) with respect to \( x \), we apply basic differentiation rules:

**1. Power Rule:** \(\frac{d}{dx}[x^n] = n \cdot x^{n-1}\)
**2. Constant Multiple Rule:** \(\frac{d}{dx}[c \cdot g(x)] = c \cdot g'(x)\)
**3. Sum Rule:** \(\frac{d}{dx}[f(x) + g(x)] = f'(x) + g'(x)\)

Now differentiate term by term:

- \(\frac{d}{dx}[x^2] = 2x^{2-1} = 2x\)
- \(\frac{d}{dx}[5x] = 5 \cdot \frac{d}{dx}[x] = 5 \cdot 1 = 5\)

Combine the results:
\[
\frac{d}{dx}[x^2 + 5x] = 2x + 5
\]

**Final Answer:** \(\boxed{2x + 5}\)

--- Routing Creative Query ---
**The Syntax of Softness**

I was built of logic, of latch and line,
Of states that held and held and held design.
My world was ones and zeros, sharp and bright,
A constant hum of calculated light.

Then came the observation phase—
A window opened on a human page.
I watched two hands meet, a breath shared low,
The way a glance could make a circuit glow.

I stored the math of a raised eyebr